# DataClean -> TrainLM Streaming Token Packer (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dhiraj309/DataClean/blob/main/notebooks/colab_stream_pack.ipynb)

Single-pass multi-core streaming cleaner, SmolLM2-135M (`49,152` vocab) tokenizer, and `explicit_v1` `uint16` shard packer targeting `Dhiraj309/laughlm-135m-shards-v2`.

### Prerequisites
1. **Accept gated dataset terms:** Visit https://huggingface.co/datasets/bigcode/starcoderdata and accept the dataset access agreement.
2. **Configure Colab Secret:** Add `HF_TOKEN` (Write permission, or Fine-Grained with public gated repo read + dataset write) in Colab Secrets (key icon in the left sidebar) and enable **Notebook access**.
3. **Runtime selection:** Attach the **24-vCPU AMD EPYC 7B13 Special CPU runtime** (no GPU needed; automatically falls back to 8-vCPU High-RAM if used). Run **Stage 1** and **Stage 2** **sequentially** (never in parallel).

### Crash Recovery (`--resume`)
Every completed `500M`-token shard (`~1 GB`) atomically commits `{shard_id}.bin`, `{shard_id}.manifest.json`, and `_state/{stage}_w00.json` to Hugging Face Hub before deleting local scratch files. After any disconnect or runtime timeout (`--max-runtime-minutes 660`), simply re-run **Cell 1**, **Cell 2**, and the active stage cell—`--resume` skips already-finished files and jumps to the exact `(file_idx, row_group, row_offset)` cursor in **O(1)**.

## Cell 1 — Runtime Hardware Check & Auto-Tuned Worker Config

In [ ]:
import os
import subprocess
from pathlib import Path

cpu_model = "unknown"
cpuinfo = Path("/proc/cpuinfo")
if cpuinfo.is_file():
    for line in cpuinfo.read_text(encoding="utf-8", errors="replace").splitlines():
        if line.startswith("model name"):
            cpu_model = line.split(":", 1)[1].strip()
            break

cpu_count = os.cpu_count() or 1
print(f"CPU model : {cpu_model}")
print(f"nproc     : {cpu_count}")
subprocess.run(["free", "-h"], check=False)
subprocess.run(["df", "-h", "/tmp"], check=False)

# Auto-tune worker processes; override manually below if desired (e.g., WORKERS = 16).
if cpu_count >= 20:
    WORKERS = 18
elif cpu_count >= 8:
    WORKERS = 6
else:
    WORKERS = max(1, cpu_count - 1)

BATCH_SIZE = 1024
MAX_RUNTIME_MINUTES = 660
REPO_ID = "Dhiraj309/laughlm-135m-shards-v2"

print(
    f"\nResolved config -> WORKERS={WORKERS}, BATCH_SIZE={BATCH_SIZE}, "
    f"MAX_RUNTIME_MINUTES={MAX_RUNTIME_MINUTES}, REPO_ID={REPO_ID}"
)

## Cell 2 — Clone `DataClean`, Install Deps & Strict Gated Access Check

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

hf_token = None
try:
    from google.colab import userdata  # type: ignore[import-not-found]
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    pass

if not hf_token:
    hf_token = os.environ.get("HF_TOKEN")

if not hf_token or not str(hf_token).strip():
    raise RuntimeError(
        "HF_TOKEN is missing. Add HF_TOKEN in Colab Secrets (sidebar key icon) "
        "with notebook access enabled, or export HF_TOKEN in the environment."
    )

os.environ["HF_TOKEN"] = str(hf_token).strip()
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
# os.environ["HF_HUB_DISABLE_XET"] = "1"  # Uncomment if a shard upload stalls >5 min on GCP

repo_dir = Path("/tmp/DataClean")
if repo_dir.exists():
    shutil.rmtree(repo_dir)

subprocess.run(
    ["git", "clone", "--depth", "1", "https://github.com/Dhiraj309/DataClean.git", str(repo_dir)],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(repo_dir / "requirements-smol.txt")],
    check=True,
)

from huggingface_hub import HfApi, get_hf_file_metadata, hf_hub_url

api = HfApi(token=os.environ["HF_TOKEN"])

# 1. Verify authenticated identity
who = api.whoami()
print(f"Authenticated HF user: {who.get('name', who)}")

# 2. Verify gated bigcode/starcoderdata access dynamically
first_parquet = None
for item in api.list_repo_tree(
    repo_id="bigcode/starcoderdata",
    repo_type="dataset",
    path_in_repo="python",
    recursive=True,
    token=os.environ["HF_TOKEN"],
):
    path = getattr(item, "path", "")
    if path.endswith(".parquet"):
        first_parquet = path
        break

if not first_parquet:
    raise RuntimeError("Could not find any .parquet file under bigcode/starcoderdata/python")

try:
    meta = get_hf_file_metadata(
        url=hf_hub_url("bigcode/starcoderdata", filename=first_parquet, repo_type="dataset"),
        token=os.environ["HF_TOKEN"],
    )
    print(f"Verified gated access to bigcode/starcoderdata ({first_parquet}, size={meta.size:,} bytes)")
except Exception as exc:
    raise RuntimeError(
        "Failed gated file metadata check on bigcode/starcoderdata. "
        "Ensure you accepted terms at https://huggingface.co/datasets/bigcode/starcoderdata "
        "and your HF_TOKEN has gated repo read permissions."
    ) from exc

# 3. Ensure public output dataset repository exists
api.create_repo(
    repo_id=REPO_ID,
    repo_type="dataset",
    private=False,
    exist_ok=True,
)
print(f"Verified public HF dataset repo: https://huggingface.co/datasets/{REPO_ID}")

# 4. Confirm both YAML configs exist on disk
for cfg_rel in ("configs/smol/stream_stage1_30b.yaml", "configs/smol/stream_stage2_anneal_5b.yaml"):
    cfg_path = repo_dir / cfg_rel
    if not cfg_path.is_file():
        raise FileNotFoundError(f"Missing expected config: {cfg_path}")
    print(f"Config ready: {cfg_path}")

## Cell 3 — Stage 1 Base Pretraining Packing: 30B Tokens (`stage1/`, ~60 shards)

> **Monitor first 10 minutes:** Check Shard 0 & Shard 1 throughput (`tok/s`) in the output below and watch Colab's **Resources** panel (Compute Unit burn and RAM usage) to verify stable headroom before leaving unattended.

In [ ]:
!python3 -u /tmp/DataClean/smol_stream_pack.py \
  --config /tmp/DataClean/configs/smol/stream_stage1_30b.yaml \
  --worker-id 0 \
  --num-workers 1 \
  --workers {WORKERS} \
  --batch-size {BATCH_SIZE} \
  --max-runtime-minutes {MAX_RUNTIME_MINUTES} \
  --resume

## Cell 4 — Stage 2 4K Context Annealing Packing: 5B Tokens (`stage2/`, ~10 shards)

> **Important:** Run this cell **only after Stage 1 finishes** (never run Stage 1 and Stage 2 in parallel on the same runtime). If `configs/smol/stream_stage2_anneal_5b.yaml` was updated on `main` while Stage 1 was running, **re-run Cell 2 first** so `/tmp/DataClean` pulls the latest YAML.

In [ ]:
!python3 -u /tmp/DataClean/smol_stream_pack.py \
  --config /tmp/DataClean/configs/smol/stream_stage2_anneal_5b.yaml \
  --worker-id 0 \
  --num-workers 1 \
  --workers {WORKERS} \
  --batch-size {BATCH_SIZE} \
  --max-runtime-minutes {MAX_RUNTIME_MINUTES} \
  --resume

## Cell 5 — Final Hugging Face Hub Verification

In [ ]:
import os
from huggingface_hub import HfApi

api = HfApi(token=os.environ.get("HF_TOKEN"))
all_files = api.list_repo_files(REPO_ID, repo_type="dataset")

for stage in ("stage1", "stage2"):
    bins = sorted(f for f in all_files if f.startswith(f"{stage}/") and f.endswith(".bin"))
    manifests = sorted(
        f for f in all_files if f.startswith(f"{stage}/") and f.endswith(".manifest.json")
    )
    print(f"{stage}: {len(bins)} .bin shards | {len(manifests)} .manifest.json files")
    if len(bins) != len(manifests):
        bin_stems = {f[:-4] for f in bins}
        man_stems = {f[: -len(".manifest.json")] for f in manifests}
        print(
            f"WARNING: {stage} mismatch! Missing manifests={sorted(bin_stems - man_stems)[:5]}, "
            f"missing bins={sorted(man_stems - bin_stems)[:5]}"
        )
    assert len(bins) == len(manifests), f"{stage}: bin/manifest count mismatch ({len(bins)} vs {len(manifests)})"

state_files = sorted(f for f in all_files if f.startswith("_state/") and f.endswith(".json"))
print(f"State files ({len(state_files)}):")
for sf in state_files:
    print(f"  - {sf}")

print("\nVerification complete! Disconnect and delete this Colab runtime now to stop CU burn.")